## Part 5 of 6: Building the Models

Loads feature-engineered df and splits from notebooks/04_feature_engineering.ipynb.

See `notebooks/README.md` for the full run order.

In [ ]:
# Import Python libraries

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (roc_auc_score, average_precision_score, f1_score, precision_recall_curve, precision_score, recall_score)
import warnings
warnings.filterwarnings("ignore")
np.random.seed(42)
print("Libraries have been imported!")

In [ ]:
# Load state saved by the previous notebook
import joblib
_state = joblib.load("_state/04_state.joblib")
globals().update(_state)
print(f"Loaded {len(_state)} objects: {sorted(_state)}")


In [ ]:
# Re-declare evaluate() and best_f1_threshold(), defined earlier in
# notebooks/04_feature_engineering.ipynb. Repeated verbatim here because
# each notebook runs in its own fresh kernel and doesn't share state
# with the notebook that defined them.
def evaluate(name, y_true, scores, threshold):
    pred = (scores >= threshold).astype(int)
    return {"model": name,
            "ROC-AUC": roc_auc_score(y_true, scores),
            "PR-AUC": average_precision_score(y_true, scores),
            "F1 (failed)": f1_score(y_true, pred),
            "Precision": precision_score(y_true, pred, zero_division=0),
            "Recall": recall_score(y_true, pred),
            "threshold": threshold}

def best_f1_threshold(y_true, scores):
    p, r, t = precision_recall_curve(y_true, scores)
    f1 = 2 * p * r / np.clip(p + r, 1e-12, None)
    return float(t[np.argmax(f1[:-1])])


In [ ]:
# Shared setup for model comparison
from sklearn.model_selection import TimeSeriesSplit

results = []
test_scores = {}

# Hyperparameter search on 5 grouped, time-split folds instead of one
# fixed validation window. Each fold's training years end strictly before
# its validation years, preventing future leakage, and any firm with a row in a
# fold's validation window is fully excluded from that fold's training rows.

years = np.sort(df.loc[df["year"] <= 2014, "year"].unique())
tscv = TimeSeriesSplit(n_splits=5, test_size=2)
FOLD_BOUNDARIES = [(years[tr_idx[0]], years[tr_idx[-1]], years[va_idx[0]], years[va_idx[-1]])
                   for tr_idx, va_idx in tscv.split(years)]

def build_grouped_time_folds(df, boundaries, features):
    folds = []
    for (tr_start, tr_end, va_start, va_end) in boundaries:
        val_window = df[df["year"].between(va_start, va_end)]
        val_firms = set(val_window["company_name"])
        train_window_raw = df[df["year"].between(tr_start, tr_end)]
        train_window_clean = train_window_raw[~train_window_raw["company_name"].isin(val_firms)]
        folds.append({
            "X_tr": train_window_clean[features], "y_tr": train_window_clean["target"],
            "X_va": val_window[features], "y_va": val_window["target"],
        })
    return folds

cv_folds_lr = build_grouped_time_folds(df, FOLD_BOUNDARIES, FEATURES_LR)
cv_folds = build_grouped_time_folds(df, FOLD_BOUNDARIES, FEATURES)


In [ ]:
# Model 1: Logistic Regression
from sklearn.linear_model import LogisticRegression

# Build five-fold cross-validation
C_grid = [0.01, 0.1, 1.0, 10.0, 100.0]
lr_cv_results = []
for C in C_grid:
    fold_scores = []
    for f in cv_folds_lr:
        scaler = StandardScaler().fit(f["X_tr"])
        Xf_tr_s, Xf_va_s = scaler.transform(f["X_tr"]), scaler.transform(f["X_va"])
        lr = LogisticRegression(max_iter=2000, C=C, class_weight="balanced", random_state=42)
        lr.fit(Xf_tr_s, f["y_tr"])
        fold_scores.append(average_precision_score(f["y_va"], lr.predict_proba(Xf_va_s)[:, 1]))
    lr_cv_results.append({"C": C, "pr_auc_mean": np.mean(fold_scores)})
    print(f"C={C}: mean PR-AUC={np.mean(fold_scores):.4f}")

# Print results of cross-validation
best_C = max(lr_cv_results, key=lambda r: r["pr_auc_mean"])["C"]
print(f"Best C: {best_C}")
# Fit the final LR with the CV-selected C on the full training window,
# freeze its threshold on validation, and score the test set once.
lr_scaler = StandardScaler().fit(X_tr_lr)
X_tr_lr_s, X_va_lr_s, X_te_lr_s = (lr_scaler.transform(X)
                                   for X in (X_tr_lr, X_va_lr, X_te_lr))
lr_final = LogisticRegression(max_iter=2000, C=best_C,
                              class_weight="balanced", random_state=42)
lr_final.fit(X_tr_lr_s, y_tr)
lr_thr = best_f1_threshold(y_va, lr_final.predict_proba(X_va_lr_s)[:, 1])
sc = lr_final.predict_proba(X_te_lr_s)[:, 1]
results.append(evaluate("LogReg (balanced)", y_te, sc, lr_thr))
test_scores["LogReg (balanced)"] = sc

# Interpretability: coefficients on standardized features
lr_coefs = pd.Series(lr_final.coef_[0], index=FEATURES_LR).sort_values()
print("Final LR coefficients (standardized features):")
print(lr_coefs.round(3).to_string())


In [ ]:
# Model 2: Full Altman Z-Score baseline
# Ranking score = -Z (lower Z = more distressed).
z_te = te["altman_z"]
results.append(evaluate("Altman Z", y_te, -z_te, -1.81))
test_scores["Altman Z"] = -z_te.values
zone = pd.cut(z_te, [-np.inf, 1.81, 2.99, np.inf],
              labels=["distress (<1.81)", "gray (1.81-2.99)", "safe (>2.99)"])
print("Test-set bankruptcy rate by Altman Z-score zone:")
print(te.groupby(zone)["target"].agg(["mean", "size"]).round(4).to_string())

In [ ]:
# Model 3: XGBoost (primary)
from xgboost import XGBClassifier

spw = (y_tr == 0).sum() / (y_tr == 1).sum()
print(f"scale_pos_weight from training data: {spw:.1f}")

grid = [{"max_depth": d, "learning_rate": lr_, "subsample": ss,
         "colsample_bytree": cs, "min_child_weight": mcw}
        for d in (3, 4, 6)
        for lr_ in (0.05, 0.1)
        for ss, cs, mcw in [(0.8, 0.8, 5), (1.0, 1.0, 1)]]

cv_results = []
for params in grid:
    fold_scores, fold_rounds = [], []
    for f in cv_folds:
        spw_f = (f["y_tr"] == 0).sum() / max((f["y_tr"] == 1).sum(), 1)
        m = XGBClassifier(n_estimators=2000, early_stopping_rounds=50,
                          scale_pos_weight=spw_f, eval_metric="aucpr",
                          tree_method="hist", random_state=42, **params)
        m.fit(f["X_tr"], f["y_tr"], eval_set=[(f["X_va"], f["y_va"])], verbose=False)
        fold_scores.append(average_precision_score(f["y_va"], m.predict_proba(f["X_va"])[:, 1]))
        fold_rounds.append(m.best_iteration + 1)
    cv_results.append({"params": params, "pr_auc_mean": np.mean(fold_scores),
                        "pr_auc_std": np.std(fold_scores), "n_rounds": int(np.median(fold_rounds))})

best_cv = max(cv_results, key=lambda r: r["pr_auc_mean"])
print(f"Best CV mean PR-AUC {best_cv['pr_auc_mean']:.4f} "
      f"(+/-{best_cv['pr_auc_std']:.4f}) with {best_cv['params']}, {best_cv['n_rounds']} rounds")

# Refit the winning hyperparameters on the original single train/val split
# so we still have one clean model that never trained on validation.
# This is what the calibration check later in the notebook relies on (best["model"]).
best = {"params": best_cv["params"], "n_rounds": best_cv["n_rounds"]}
tuning_fit = XGBClassifier(n_estimators=best["n_rounds"], scale_pos_weight=spw,
                           eval_metric="aucpr", tree_method="hist",
                           random_state=42, **best["params"])
tuning_fit.fit(X_tr, y_tr, verbose=False)
best["model"] = tuning_fit
best["pr_auc"] = average_precision_score(y_va, tuning_fit.predict_proba(X_va)[:, 1])

# Threshold frozen from val set predictions of the tuning-stage model,
# so it is selected on data the final model never trained on.
xgb_thr = best_f1_threshold(y_va, best["model"].predict_proba(X_va)[:, 1])

# Retrain on combined 1999-2014 with chosen hyperparameters and the round
# count found by early stopping; evaluate on 2015-2018 test set.
X_trva = pd.concat([X_tr, X_va]); y_trva = pd.concat([y_tr, y_va])
spw_full = (y_trva == 0).sum() / (y_trva == 1).sum()
xgb = XGBClassifier(n_estimators=best["n_rounds"], scale_pos_weight=spw_full,
                    eval_metric="aucpr", tree_method="hist",
                    random_state=42, **best["params"])
xgb.fit(X_trva, y_trva, verbose=False)
sc = xgb.predict_proba(X_te)[:, 1]
results.append(evaluate("XGBoost (primary)", y_te, sc, xgb_thr))
test_scores["XGBoost (primary)"] = sc

In [ ]:
# Also trying XGBoost with SMOTE (comparison against scale_pos_weight)
from imblearn.over_sampling import SMOTE

X_tr_sm, y_tr_sm = SMOTE(random_state=42).fit_resample(X_tr, y_tr)
print(f"Training set size before SMOTE: {len(y_tr):,} ({y_tr.mean():.3%} positive)")
print(f"Training set size after SMOTE:  {len(y_tr_sm):,} ({y_tr_sm.mean():.3%} positive)")

xgb_sm = XGBClassifier(n_estimators=best["n_rounds"],
                        scale_pos_weight=1,
                        eval_metric="aucpr",
                        tree_method="hist",
                        random_state=42,
                        **best["params"])
xgb_sm.fit(X_tr_sm, y_tr_sm, verbose=False)

# Threshold selected on the unmodified validation set.
sm_thr = best_f1_threshold(y_va, xgb_sm.predict_proba(X_va)[:, 1])
sc = xgb_sm.predict_proba(X_te)[:, 1]
results.append(evaluate("XGBoost (SMOTE)", y_te, sc, sm_thr))
test_scores["XGBoost (SMOTE)"] = sc

In [ ]:
# Model 4: Decision Tree (for interpretability) and Model 5: Random Forest
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier

# Decision Tree: shallow and class-weighted to avoid class imbalance
dt = DecisionTreeClassifier(max_depth=3, class_weight="balanced", random_state=42)
dt.fit(X_tr_dt, y_tr)
dt_thr = best_f1_threshold(y_va, dt.predict_proba(X_va_dt)[:, 1])
sc = dt.predict_proba(X_te_dt)[:, 1]
results.append(evaluate("Decision Tree (depth=3)", y_te, sc, dt_thr))
test_scores["Decision Tree (depth=3)"] = sc

# Plot decision tree structure
plt.figure(figsize=(20, 9))
plot_tree(dt, feature_names=FEATURES_DT, class_names=["alive", "failed"],
          filled=True, rounded=True, fontsize=8, proportion=True)
plt.title("Decision tree structure (depth=3)")
plt.tight_layout(); plt.show()

# Model 5: Random Forest
rf = RandomForestClassifier(n_estimators=500, class_weight="balanced_subsample",
                            random_state=42, n_jobs=-1)
rf.fit(X_tr, y_tr)
rf_thr = best_f1_threshold(y_va, rf.predict_proba(X_va)[:, 1])
sc = rf.predict_proba(X_te)[:, 1]
results.append(evaluate("Random Forest", y_te, sc, rf_thr))
test_scores["Random Forest"] = sc

rf_imp = pd.Series(rf.feature_importances_, index=FEATURES).sort_values(ascending=False)
print("Random Forest top 10 features:")
print(rf_imp.head(10).round(4).to_string())

In [ ]:
# Save state for the next notebook
import joblib
from pathlib import Path
Path("_state").mkdir(exist_ok=True)
joblib.dump({
    "results": results, "test_scores": test_scores, "best": best, "xgb": xgb, "xgb_thr": xgb_thr, "X_trva": X_trva, "X_te": X_te, "X_va": X_va, "y_va": y_va, "y_te": y_te, "te": te, "FEATURES": FEATURES
}, "_state/05_state.joblib")
